# EXPERIMENT–2: Text Retrieval and Text Processing

**Tasks:**
1. Obtain a Twitter/X tweet dataset from Kaggle and load it into Colab.
2. Extract the first 100 tweets and save them to a file.
3. Perform sentence tokenization.
4. Perform word tokenization.
5. Perform basic tweet cleaning and word-frequency analysis.


In [2]:
!pip install -q nltk pandas

import os
import re
import glob
import pandas as pd
from collections import Counter
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## 2. Upload the Kaggle tweet dataset

In [3]:
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

Saving tweets.csv to tweets.csv
Uploaded files: ['tweets.csv']


## 3. Locate and read the CSV file

In [5]:
csv_files = glob.glob('*.csv')
if not csv_files:
    raise FileNotFoundError('No CSV file found. Please upload the Kaggle CSV dataset.')

print('CSV files found:')
for f in csv_files:
    print(' -', f)

csv_file = csv_files[0]
df = pd.read_csv(csv_file)

print('\nUsing:', csv_file)
print('Dataset shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

display(df.head())

CSV files found:
 - tweets.csv

Using: tweets.csv
Dataset shape: (400, 2)

Columns:
['text', 'date']


,text,date
0,Ready for the festivities that are yet to come...,29/10/2023
1,"Lights, camera, action. Let’s get your style s...",24/10/2023
2,Radiant in red.\n\nPlay with print this season...,18/10/2023
3,Paint your own story where the sea meets the g...,14/10/2023
4,The Formal Edit – celebrate in the colours of ...,20/09/2023


## 4. Identify the tweet/text column

In [6]:
common_text_columns = [
    'text', 'tweet', 'tweets', 'content', 'full_text',
    'Tweet', 'Tweets', 'Text', 'Content', 'fullText'
]

text_column = None
for col in common_text_columns:
    if col in df.columns:
        text_column = col
        break

if text_column is None:
    # Fallback: choose the object/string column with the largest average length.
    object_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
    if not object_cols:
        raise ValueError('No text/string column was found. Please inspect the dataset columns.')
    avg_lengths = {c: df[c].dropna().astype(str).str.len().mean() for c in object_cols}
    text_column = max(avg_lengths, key=avg_lengths.get)

print('Selected text column:', text_column)
print('\nSample text:')
print(df[text_column].dropna().iloc[0])

Selected text column: text

Sample text:
Ready for the festivities that are yet to come, she writes to her loved ones while dressed in the rich tones of winter. Wrapped in ready-to-wear fits of raw silk and jacquard with damask-inspired prints and embroideries.

#ReadyToWear #FestiveWear #Khaadi #WearYourself


## 5. Extract the first 100 tweets

In [7]:
tweets = df[text_column].dropna().astype(str).head(100).reset_index(drop=True)

print('Number of tweets extracted:', len(tweets))

display(pd.DataFrame({'Tweet No.': range(1, len(tweets)+1), 'Tweet': tweets}))

Number of tweets extracted: 100


,Tweet No.,Tweet
0,1,Ready for the festivities that are yet to come...
1,2,"Lights, camera, action. Let’s get your style s..."
2,3,Radiant in red.\n\nPlay with print this season...
3,4,Paint your own story where the sea meets the g...
4,5,The Formal Edit – celebrate in the colours of ...
...,...,...
95,96,We’ve opened our third & largest Khaadi Outlet...
96,97,"One fit, so many ways to slay.\n\nAvailable in..."
97,98,"Mix, match and slay!\n\n#WinterWear #Khaadi"
98,99,"Cozy, but make it cool.\n\nAvailable in-stores..."


## 6. Save the first 100 tweets to a CSV file

In [8]:
first_100 = pd.DataFrame({
    'tweet_id': range(1, len(tweets) + 1),
    'text': tweets
})

first_100.to_csv('first_100_tweets.csv', index=False)
print('Saved as first_100_tweets.csv')

files.download('first_100_tweets.csv')

Saved as first_100_tweets.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Sentence Tokenization

In [9]:
sample_text = 'Natural Language Processing is a field of Artificial Intelligence. It helps computers understand human language.'
sentences = sent_tokenize(sample_text)

print('Sample text:\n', sample_text)
print('\nSentence tokens:')
for i, sentence in enumerate(sentences, 1):
    print(f'{i}. {sentence}')

Sample text:
 Natural Language Processing is a field of Artificial Intelligence. It helps computers understand human language.

Sentence tokens:
1. Natural Language Processing is a field of Artificial Intelligence.
2. It helps computers understand human language.


## 8. Word Tokenization

In [10]:
words = word_tokenize(sample_text)

print('Word tokens:')
print(words)
print('\nNumber of tokens:', len(words))

Word tokens:
['Natural', 'Language', 'Processing', 'is', 'a', 'field', 'of', 'Artificial', 'Intelligence', '.', 'It', 'helps', 'computers', 'understand', 'human', 'language', '.']

Number of tokens: 17


## 9. Sentence Tokenization of the first 100 tweets

In [11]:
tweet_sentences = []

for tweet_no, tweet in enumerate(tweets, 1):
    for sentence in sent_tokenize(tweet):
        tweet_sentences.append((tweet_no, sentence))

sentence_df = pd.DataFrame(tweet_sentences, columns=['tweet_id', 'sentence'])

print('Total sentences:', len(sentence_df))
display(sentence_df.head(20))

Total sentences: 346


,tweet_id,sentence
0,1,Ready for the festivities that are yet to come...
1,1,Wrapped in ready-to-wear fits of raw silk and ...
2,1,#ReadyToWear #FestiveWear #Khaadi #WearYourself
3,2,"Lights, camera, action."
4,2,Let’s get your style sorted ft. Sabeena Farooq.
5,2,Let Sabeena be your chaperone for all your AM-...
6,2,Available in-store and online on 27th October.
7,2,#ADayInTheLifeOfSabeenaFarooq #SabeenaFarooq #...
8,3,Radiant in red.
9,3,Play with print this season and create your vi...


## 10. Word Tokenization of the first 100 tweets

In [12]:
tweet_words = []

for tweet_no, tweet in enumerate(tweets, 1):
    tokens = word_tokenize(tweet)
    for token in tokens:
        tweet_words.append((tweet_no, token))

word_df = pd.DataFrame(tweet_words, columns=['tweet_id', 'token'])

print('Total word tokens:', len(word_df))
display(word_df.head(100))

Total word tokens: 3649


,tweet_id,token
0,1,Ready
1,1,for
2,1,the
3,1,festivities
4,1,that
...,...,...
95,2,WearYourself
96,3,Radiant
97,3,in
98,3,red


## 11. Basic Tweet Cleaning

In [13]:
def clean_tweet(text):
    text = re.sub(r'https?://\\S+|www\\.\\S+', '', text)  # URLs
    text = re.sub(r'@\\w+', '', text)                       # mentions
    text = re.sub(r'#', '', text)                            # hashtag symbol
    text = re.sub(r'\\s+', ' ', text).strip()               # extra spaces
    return text

processed_df = first_100.copy()
processed_df['cleaned_text'] = processed_df['text'].apply(clean_tweet)

display(processed_df.head(10))

,tweet_id,text,cleaned_text
0,1,Ready for the festivities that are yet to come...,Ready for the festivities that are yet to come...
1,2,"Lights, camera, action. Let’s get your style s...","Lights, camera, action. Let’s get your style s..."
2,3,Radiant in red.\n\nPlay with print this season...,Radiant in red.\n\nPlay with print this season...
3,4,Paint your own story where the sea meets the g...,Paint your own story where the sea meets the g...
4,5,The Formal Edit – celebrate in the colours of ...,The Formal Edit – celebrate in the colours of ...
5,6,Artist and aspiring filmmaker SamioRenalda roc...,Artist and aspiring filmmaker SamioRenalda roc...
6,7,Portrait photographer and lifestyle content cr...,Portrait photographer and lifestyle content cr...
7,8,What a perfect me-time looks like.\n\nFeel the...,What a perfect me-time looks like.\n\nFeel the...
8,9,Spontaneous and lively with a burst of freshne...,Spontaneous and lively with a burst of freshne...
9,10,"In the midst of the bustle, she is lost in her...","In the midst of the bustle, she is lost in her..."


## 12. Word Frequency Analysis

In [14]:
all_words = []

for tweet in processed_df['cleaned_text']:
    tokens = word_tokenize(tweet.lower())
    all_words.extend(tokens)

# Keep only alphanumeric tokens
words_without_punctuation = [word for word in all_words if word.isalnum()]

frequency = Counter(words_without_punctuation)

print('Top 20 words/tokens:')
for word, count in frequency.most_common(20):
    print(f'{word}: {count}')

Top 20 words/tokens:
and: 134
the: 122
khaadi: 114
wearyourself: 79
of: 58
in: 53
to: 49
with: 47
a: 46
your: 41
for: 37
is: 35
collection: 34
you: 34
it: 34
our: 31
s: 30
available: 27
online: 26
fabrics: 25


## 13. Save the processed data

In [15]:
processed_df.to_csv('processed_tweets.csv', index=False)
sentence_df.to_csv('sentence_tokens.csv', index=False)
word_df.to_csv('word_tokens.csv', index=False)

print('Files created:')
print('1. first_100_tweets.csv')
print('2. processed_tweets.csv')
print('3. sentence_tokens.csv')
print('4. word_tokens.csv')

Files created:
1. first_100_tweets.csv
2. processed_tweets.csv
3. sentence_tokens.csv
4. word_tokens.csv


## 14. Download the output files

In [16]:
from google.colab import files

for output_file in [
    'first_100_tweets.csv',
    'processed_tweets.csv',
    'sentence_tokens.csv',
    'word_tokens.csv'
]:
    files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>